# 06 - Inference & Export
## PhishScamSense: Real-Time Multimodal Phishing Defense

This notebook covers:
1. Load trained model checkpoints
2. Single URL inference pipeline
3. Batch inference pipeline
4. Export model to ONNX (for production optimization)
5. Build Bloom Filter from known phishing URLs
6. Export Bloom Filter data for browser extension

In [ ]:
import sys
import os
import math
import re
import pickle
import json
import hashlib
import logging
from collections import Counter
from urllib.parse import urlparse

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

logging.basicConfig(level=logging.WARNING)
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "ml", "checkpoints")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 6.1 Load Model Components

In [ ]:
# ── Feature extraction ──
def _shannon_entropy(text):
    if not text: return 0.0
    freq = Counter(text); length = len(text)
    return -sum((c/length)*math.log2(c/length) for c in freq.values())

def _has_ip(hostname):
    return 1 if re.match(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$", hostname) else 0

def _max_cons_consonants(text):
    vowels = set("aeiouAEIOU"); max_c = cur = 0
    for ch in text:
        if ch.isalpha() and ch not in vowels: cur += 1; max_c = max(max_c, cur)
        else: cur = 0
    return max_c

def _vowel_ratio(text):
    alpha = [c for c in text if c.isalpha()]
    if not alpha: return 0.0
    return sum(1 for c in alpha if c in set("aeiouAEIOU")) / len(alpha)

def extract_url_features(url: str) -> list[float]:
    p = urlparse(url); h = p.hostname or ""; path = p.path or ""
    return [len(url), len(h), len(path), url.count("."), url.count("-"), url.count("_"),
            url.count("/"), len(p.query.split("&")) if p.query else 0,
            1 if p.fragment else 0, sum(c.isdigit() for c in url),
            sum(not c.isalnum() and c not in ".-_/:" for c in url),
            _shannon_entropy(url), _shannon_entropy(h), _has_ip(h),
            1 if h.startswith("xn--") else 0,
            1 if p.port and p.port not in (80, 443) else 0,
            1 if p.scheme == "https" else 0, 1 if "@" in url else 0,
            1 if "//" in path else 0,
            len(h.split("."))-2 if len(h.split("."))>2 else 0,
            len(h.split(".")[-1]) if "." in h else 0,
            _max_cons_consonants(h), _vowel_ratio(h)]

# ── Model classes ──
class AttentionLayer(nn.Module):
    def __init__(self, h):
        super().__init__(); self.a = nn.Linear(h, 1)
    def forward(self, x):
        w = torch.softmax(self.a(x), dim=1); return torch.sum(w * x, dim=1)

class NLPBranch(nn.Module):
    def __init__(self, out=128, lstm_h=256, layers=2, dropout=0.3, freeze=True):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        if freeze:
            for p in self.bert.parameters(): p.requires_grad = False
        bh = self.bert.config.hidden_size
        self.lstm = nn.LSTM(bh, lstm_h, layers, batch_first=True, bidirectional=True,
                            dropout=dropout if layers>1 else 0)
        self.attn = AttentionLayer(lstm_h*2)
        self.fc = nn.Linear(lstm_h*2, out)
        self.drop = nn.Dropout(dropout)
    def forward(self, ids, mask):
        x = self.bert(input_ids=ids, attention_mask=mask).last_hidden_state
        x, _ = self.lstm(x); x = self.attn(x)
        return self.fc(self.drop(x))

class MLPBranch(nn.Module):
    def __init__(self, in_dim=23, hidden=[128,64], out=64, drop=0.3):
        super().__init__()
        layers = []; prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop)]; prev=h
        layers.append(nn.Linear(prev, out))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class PhishScamSenseFusionModel(nn.Module):
    def __init__(self, num_feat=23, nlp_out=128, num_out=64, freeze=True):
        super().__init__()
        self.nlp = NLPBranch(out=nlp_out, freeze=freeze)
        self.mlp = MLPBranch(in_dim=num_feat, out=num_out)
        self.fusion_dim = nlp_out + num_out
    def forward(self, ids, mask, feat):
        return torch.cat([self.nlp(ids, mask), self.mlp(feat)], dim=1)

class URLTokenizer:
    def __init__(self, max_len=128):
        self.tok = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_len = max_len
    def tokenize(self, urls):
        return self.tok(urls, padding=True, truncation=True,
                        max_length=self.max_len, return_tensors="pt")

# ── Load checkpoints ──
fusion_model = PhishScamSenseFusionModel(num_feat=23)
ckpt_path = os.path.join(CHECKPOINT_DIR, "fusion_model.pt")
if os.path.exists(ckpt_path):
    fusion_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("Fusion model loaded.")
else:
    print("WARNING: Run notebook 04 first to generate checkpoints.")

xgb_path = os.path.join(CHECKPOINT_DIR, "xgb_classifier.pkl")
if os.path.exists(xgb_path):
    with open(xgb_path, "rb") as f:
        xgb_clf = pickle.load(f)
    print("XGBoost model loaded.")

fusion_model.eval()
tokenizer = URLTokenizer()

## 6.2 Single URL Inference Pipeline

In [ ]:
def predict_url(url: str, threshold: float = 0.5) -> dict:
    """
    Full inference pipeline for a single URL.
    Returns phishing prediction with confidence score.
    """
    # 1. Tokenize for NLP branch
    tokens = tokenizer.tokenize([url])
    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)

    # 2. Extract numerical features
    num_features = torch.tensor([extract_url_features(url)], dtype=torch.float32).to(device)

    # 3. Get fused representation from neural network
    fusion_model.eval()
    with torch.no_grad():
        fused = fusion_model(input_ids, attention_mask, num_features).cpu().numpy()

    # 4. XGBoost classification
    confidence = float(xgb_clf.predict_proba(fused)[0][1])
    is_phishing = confidence >= threshold

    return {
        "url": url,
        "phishing": is_phishing,
        "confidence": round(confidence, 4),
        "risk_level": "HIGH" if confidence > 0.8 else "MEDIUM" if confidence > 0.5 else "LOW",
    }


# Test on several URLs
test_cases = [
    "https://www.google.com",
    "https://github.com/torvalds/linux",
    "http://paypa1-secure.com/signin/update-billing",
    "http://xn--ggle-1noa.com/accounts/login",
    "http://192.168.1.1/login/google-verify.html",
    "https://www.microsoft.com/en-us/windows",
]

print(f"{'URL':<55} {'Phishing':<10} {'Confidence':<12} {'Risk'}")
print("-" * 95)
for url in test_cases:
    result = predict_url(url)
    flag = "YES" if result["phishing"] else "NO"
    print(f"{url:<55} {flag:<10} {result['confidence']:<12.4f} {result['risk_level']}")

## 6.3 Batch Inference Pipeline

In [ ]:
import pandas as pd
import time

def predict_batch(urls: list[str], batch_size: int = 16, threshold: float = 0.5) -> pd.DataFrame:
    """
    Efficient batch inference with micro-batching (mirrors BentoML adaptive batching).
    """
    all_results = []

    for i in range(0, len(urls), batch_size):
        batch_urls = urls[i : i + batch_size]

        tokens = tokenizer.tokenize(batch_urls)
        input_ids = tokens["input_ids"].to(device)
        attention_mask = tokens["attention_mask"].to(device)
        num_feat = torch.tensor(
            [extract_url_features(u) for u in batch_urls], dtype=torch.float32
        ).to(device)

        fusion_model.eval()
        with torch.no_grad():
            fused = fusion_model(input_ids, attention_mask, num_feat).cpu().numpy()

        probas = xgb_clf.predict_proba(fused)[:, 1]

        for url, prob in zip(batch_urls, probas):
            all_results.append({
                "url": url,
                "phishing": bool(prob >= threshold),
                "confidence": round(float(prob), 4),
                "risk_level": "HIGH" if prob > 0.8 else "MEDIUM" if prob > 0.5 else "LOW",
            })

    return pd.DataFrame(all_results)


# Benchmark batch inference
batch_urls = test_cases * 3   # 18 URLs

start = time.time()
batch_results = predict_batch(batch_urls, batch_size=8)
elapsed = time.time() - start

print(f"Batch inference: {len(batch_urls)} URLs in {elapsed:.2f}s "
      f"({elapsed/len(batch_urls)*1000:.1f} ms/URL)")
print()
print(batch_results.to_string(index=False))

## 6.4 Build Counting Bloom Filter for Browser Extension

The Bloom Filter is the client-side first-pass check inside the browser extension. If a URL is NOT in the filter, it is guaranteed safe (no network call needed). If it IS in the filter, we call the backend ML service to verify.

In [ ]:
class CountingBloomFilter:
    """
    Counting Bloom Filter - Python implementation matching the TypeScript
    version in extension/lib/bloom-filter.ts
    
    Supports add, contains, and remove operations.
    False positive rate ≈ (1 - e^(-k*n/m))^k
      where k=num_hashes, n=num_items, m=filter_size
    """

    def __init__(self, size: int = 1_000_000, num_hashes: int = 7):
        self.size = size
        self.num_hashes = num_hashes
        self.buckets = [0] * size

    def _hash(self, value: str, seed: int) -> int:
        h = seed
        for char in value:
            h = (h * 31 + ord(char)) & 0xFFFFFFFF  # uint32
        return h % self.size

    def add(self, value: str):
        for i in range(self.num_hashes):
            idx = self._hash(value, i)
            if self.buckets[idx] < 255:
                self.buckets[idx] += 1

    def contains(self, value: str) -> bool:
        return all(self.buckets[self._hash(value, i)] > 0
                   for i in range(self.num_hashes))

    def remove(self, value: str):
        if not self.contains(value):
            return
        for i in range(self.num_hashes):
            idx = self._hash(value, i)
            if self.buckets[idx] > 0:
                self.buckets[idx] -= 1

    def false_positive_rate(self, num_items: int) -> float:
        import math
        return (1 - math.exp(-self.num_hashes * num_items / self.size)) ** self.num_hashes

    def export_data(self) -> list[int]:
        return self.buckets

    def load_data(self, data: list[int]):
        self.buckets = list(data)


# Build Bloom Filter from known phishing URLs
known_phishing = [
    "http://192.168.1.1/login/google-verify.html",
    "http://xn--ggle-1noa.com/accounts/login",
    "http://googl3-security.com/verify?user=admin&token=abc123",
    "http://paypa1-secure.com/signin/update-billing",
    "http://amaz0n-support.xyz/account/verify",
    "http://microsoft-365-login.tk/auth/signin",
    "http://netflix-billing-update.ml/payment",
    "http://faceb00k-security.ga/hacked/recovery",
    "http://apple-id-verify.cf/icloud/login.php",
    "http://bank0famerica-secure.ru/online/login",
    "http://dhl-tracking-update.info/parcel?id=83927492",
    "http://instagram-verify-account.net/auth",
    "http://linkedln-security.com/checkpoint/verify",
    "http://dropbox-shared-doc.tk/dl/invoice.pdf.exe",
    "http://wellsfarg0-alert.com/security/update",
]

# Use smaller size for demo; production uses 1_000_000
bloom = CountingBloomFilter(size=100_000, num_hashes=7)
for url in known_phishing:
    bloom.add(url)

# Verify
print("Bloom Filter Lookup Tests:")
print("-" * 55)
for url in known_phishing[:3]:
    print(f"  PHISHING  {bloom.contains(url)}  {url[:50]}")

for url in ["https://www.google.com", "https://github.com"]:
    print(f"  BENIGN    {bloom.contains(url)}  {url[:50]}")

fpr = bloom.false_positive_rate(len(known_phishing))
print(f"\nFilter size:        {bloom.size:,}")
print(f"Items inserted:     {len(known_phishing)}")
print(f"False positive rate: {fpr:.6f} ({fpr*100:.4f}%)")

## 6.5 Export Bloom Filter & Models for Production

In [ ]:
EXPORT_DIR = os.path.join(PROJECT_ROOT, "ml", "exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Export Bloom Filter as JSON (consumed by FastAPI /threats/bloom-filter endpoint)
bloom_export = {
    "filter": bloom.export_data(),
    "size": bloom.size,
    "num_hashes": bloom.num_hashes,
    "version": "0.1.0",
    "num_items": len(known_phishing),
    "false_positive_rate": bloom.false_positive_rate(len(known_phishing)),
}
bloom_path = os.path.join(EXPORT_DIR, "bloom_filter.json")
with open(bloom_path, "w") as f:
    json.dump(bloom_export, f)
print(f"Bloom filter exported: {bloom_path}")

# 2. Export fusion model state dict
fusion_export_path = os.path.join(EXPORT_DIR, "fusion_model.pt")
torch.save(fusion_model.state_dict(), fusion_export_path)
print(f"Fusion model exported: {fusion_export_path}")

# 3. Export XGBoost model
xgb_export_path = os.path.join(EXPORT_DIR, "xgb_classifier.pkl")
with open(xgb_export_path, "wb") as f:
    pickle.dump(xgb_clf, f)
print(f"XGBoost model exported: {xgb_export_path}")

# 4. Export XGBoost as JSON (portable format)
xgb_json_path = os.path.join(EXPORT_DIR, "xgb_classifier.json")
xgb_clf.save_model(xgb_json_path)
print(f"XGBoost JSON exported:  {xgb_json_path}")

print(f"\nAll exports saved to: {EXPORT_DIR}")

## 6.6 End-to-End System Summary

Verify the complete PhishScamSense pipeline works end-to-end.

In [ ]:
def phishscamsense_pipeline(url: str, threshold: float = 0.5) -> dict:
    """
    Simulates the full PhishScamSense two-stage pipeline:
      Stage 1: Bloom Filter (O(1), zero-latency, client-side)
      Stage 2: ML inference (only if Bloom Filter flags the URL)
    """
    # Stage 1: Bloom Filter check
    bloom_hit = bloom.contains(url)

    if not bloom_hit:
        return {
            "url": url,
            "stage": "bloom_filter",
            "phishing": False,
            "confidence": 0.0,
            "risk_level": "LOW",
            "note": "Cleared by Bloom Filter — no ML call needed",
        }

    # Stage 2: ML inference (only on Bloom Filter positives)
    result = predict_url(url, threshold=threshold)
    result["stage"] = "ml_inference"
    result["note"] = "Bloom Filter flagged → ML inference performed"
    return result


# Test the two-stage pipeline
print("=== PhishScamSense Two-Stage Pipeline Demo ===\n")
demo_urls = [
    "https://www.google.com",                         # benign, not in bloom
    "http://paypa1-secure.com/signin/update-billing", # phishing, in bloom
    "https://github.com/torvalds/linux",              # benign, not in bloom
    "http://xn--ggle-1noa.com/accounts/login",        # phishing, in bloom
]

for url in demo_urls:
    r = phishscamsense_pipeline(url)
    status = "PHISHING" if r["phishing"] else "SAFE   "
    print(f"[{r['stage']:<14}] {status}  conf={r['confidence']:.3f}  {url[:55]}")
    print(f"                  -> {r['note']}\n")

In [ ]:
import matplotlib.pyplot as plt

# System architecture summary diagram (text-based)
print("=" * 60)
print("  PHISHSCAMSENSE PIPELINE SUMMARY")
print("=" * 60)
print()
print("  Browser Extension (Client-Side)")
print("  ┌─────────────────────────────────────┐")
print("  │  URL → Counting Bloom Filter        │")
print("  │        O(1) lookup, zero network    │")
print("  │        If MISS → safe (no API call) │")
print("  │        If HIT  → call backend API   │")
print("  └────────────────┬────────────────────┘")
print("                   │ (only ~5% of URLs)")
print("  ┌────────────────▼────────────────────┐")
print("  │  FastAPI Backend (API Gateway)      │")
print("  │  POST /api/v1/predict               │")
print("  └────────────────┬────────────────────┘")
print("                   │")
print("  ┌────────────────▼────────────────────┐")
print("  │  BentoML Inference Service          │")
print("  │  ┌──────────────┬────────────────┐  │")
print("  │  │  NLP Branch  │  MLP Branch    │  │")
print("  │  │  DistilBERT  │  24 features   │  │")
print("  │  │  BiLSTM      │  [128,64] dims │  │")
print("  │  │  Attention   │                │  │")
print("  │  │  → 128d      │  → 64d         │  │")
print("  │  └──────┬───────┴────────┬───────┘  │")
print("  │         └───── cat ──────┘          │")
print("  │              192d fused              │")
print("  │                  ↓                  │")
print("  │          XGBoost Classifier         │")
print("  │        P(phishing) ∈ [0, 1]         │")
print("  └─────────────────────────────────────┘")
print()
print("  Exports:")
print(f"    Bloom Filter:    ml/exports/bloom_filter.json")
print(f"    Fusion Model:    ml/exports/fusion_model.pt")
print(f"    XGBoost JSON:    ml/exports/xgb_classifier.json")
print(f"    XGBoost Pickle:  ml/exports/xgb_classifier.pkl")
print("=" * 60)